## Required Imports

In [ ]:
# notebooks/08_matchup_analysis.ipynb

import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load all required data
vulnerability_df = pd.read_parquet(
    '../data/processed/batter_vulnerability.parquet'
)
pitcher_profiles = pd.read_parquet(
    '../data/processed/pitcher_profiles.parquet'
)
hitter_count_vuln = pd.read_parquet(
    '../data/processed/hitter_count_vulnerability.parquet'
)

# Load full data with recommendations
data = pd.read_parquet(
    '../data/processed/pitcher_data_with_recommendations.parquet'
)

print("All Phase 4 data loaded successfully")
print(f"Batters profiled: {vulnerability_df['batter'].nunique()}")
print(f"Pitchers profiled: {len(pitcher_profiles)}")

## Build the Matchup Engine

In [ ]:
def calculate_matchup_score(pitcher_name, batter_id, stand, data, 
                             vulnerability_df):
    """
    Calculate a matchup exploitation score for a specific
    pitcher-batter combination.
    
    Higher score = batter is better positioned to exploit 
    pitcher's deviation tendencies.
    """
    
    # Get pitcher's deviation substitution patterns
    pitcher_data = data[data['pitcher_name'] == pitcher_name].copy()
    deviations = pitcher_data[
        ~pitcher_data['followed_recommendation']
    ].copy()
    
    if len(deviations) == 0:
        return None
    
    # What does this pitcher actually throw when deviating?
    deviation_pitches = (
        deviations['pitch_type']
        .value_counts(normalize=True)
        .reset_index()
    )
    deviation_pitches.columns = ['pitch_type', 'deviation_frequency']
    
    # Get batter vulnerability against those pitches
    batter_vuln = vulnerability_df[
        (vulnerability_df['batter'] == batter_id) &
        (vulnerability_df['stand'] == stand)
    ][['pitch_type', 'vulnerability_score']]
    
    # Invert vulnerability — lower vulnerability = better for batter
    batter_vuln = batter_vuln.copy()
    batter_vuln['exploitation_score'] = (
        100 - batter_vuln['vulnerability_score']
    )
    
    # Merge deviation frequency with batter exploitation score
    matchup = deviation_pitches.merge(
        batter_vuln, on='pitch_type', how='inner'
    )
    
    if len(matchup) == 0:
        return None
    
    # Weighted matchup score
    matchup['weighted_score'] = (
        matchup['deviation_frequency'] * 
        matchup['exploitation_score']
    )
    
    total_score = matchup['weighted_score'].sum()
    
    return {
        'pitcher_name': pitcher_name,
        'batter_id': batter_id,
        'matchup_score': round(total_score, 2),
        'pitch_breakdown': matchup
    }

print("Matchup engine defined successfully")

## Run All Matchups

In [ ]:
# Calculate matchup scores for all pitcher-batter combinations
matchup_records = []

pitchers = data['pitcher_name'].unique()
batters = vulnerability_df[['batter', 'batter_name', 'stand']].drop_duplicates()

print(f"Calculating matchups for {len(pitchers)} pitchers "
      f"x {len(batters)} batters...")

for pitcher in pitchers:
    for _, batter_row in batters.iterrows():
        result = calculate_matchup_score(
            pitcher,
            batter_row['batter'],
            batter_row['stand'],
            data,
            vulnerability_df
        )
        if result:
            matchup_records.append({
                'pitcher_name': result['pitcher_name'],
                'batter_id': batter_row['batter'],
                'batter_name': batter_row['batter_name'],
                'stand': batter_row['stand'],
                'matchup_score': result['matchup_score']
            })

matchup_df = pd.DataFrame(matchup_records)
print(f"Total matchups calculated: {len(matchup_df)}")
print(f"\nSample matchup scores:")
print(matchup_df.sort_values(
    'matchup_score', ascending=False
).head(10).to_string(index=False))

## Find Most Dangerous Matchups Per Pitcher

In [ ]:
print("=== Most Dangerous Matchups by Pitcher ===\n")

for pitcher in sorted(pitcher_profiles['pitcher_name'].unique()):
    pitcher_matchups = matchup_df[
        matchup_df['pitcher_name'] == pitcher
    ].sort_values('matchup_score', ascending=False)
    
    print(f"{pitcher} — Top 5 Most Dangerous Batters:")
    print(pitcher_matchups[['batter_name', 'stand', 'matchup_score']]
          .head(5).to_string(index=False))
    print()

## Matchup Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Overall distribution of matchup scores
axes[0].hist(matchup_df['matchup_score'], bins=50, 
             color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(matchup_df['matchup_score'].mean(), 
                color='red', linestyle='--', linewidth=2,
                label=f"Mean: {matchup_df['matchup_score'].mean():.1f}")
axes[0].set_title("Distribution of All Matchup Scores", fontsize=12)
axes[0].set_xlabel("Matchup Score")
axes[0].set_ylabel("Count")
axes[0].legend()

# Average matchup score by pitcher — who is most exploitable overall
pitcher_avg = (matchup_df.groupby('pitcher_name')['matchup_score']
               .mean()
               .sort_values(ascending=True))

axes[1].barh(pitcher_avg.index, pitcher_avg.values, 
             color='steelblue', alpha=0.8)
axes[1].axvline(pitcher_avg.mean(), color='red', linestyle='--',
                linewidth=2, label=f"Mean: {pitcher_avg.mean():.1f}")
axes[1].set_title("Average Matchup Exploitation Score by Pitcher\n"
                  "Higher = more batters can exploit deviations", fontsize=12)
axes[1].set_xlabel("Average Matchup Score")
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/figures/matchup_distribution.png',
            dpi=150, bbox_inches='tight')
plt.show()

print("=== Average Exploitability by Pitcher ===")
print(pitcher_avg.sort_values(ascending=False).round(2))

## Handedness Split Analysis

In [ ]:
# Do left-handed batters exploit deviation patterns differently?
hand_matchup = matchup_df.groupby(
    ['pitcher_name', 'stand']
)['matchup_score'].mean().reset_index()

hand_pivot = hand_matchup.pivot(
    index='pitcher_name',
    columns='stand',
    values='matchup_score'
).round(2)

hand_pivot['handedness_gap'] = (
    hand_pivot.get('L', 0) - hand_pivot.get('R', 0)
).round(2)

print("=== Average Matchup Score by Batter Handedness ===")
print("Positive gap = lefties exploit deviations more")
print("Negative gap = righties exploit deviations more\n")
print(hand_pivot.sort_values('handedness_gap', ascending=False)
      .to_string())

fig, ax = plt.subplots(figsize=(12, 7))
hand_pivot[['L', 'R']].plot(kind='bar', ax=ax, 
                              color=['#d62728', '#1f77b4'],
                              alpha=0.8, width=0.6)
ax.set_title("Matchup Exploitation Score by Batter Handedness\n"
             "Per Pitcher", fontsize=12)
ax.set_ylabel("Average Matchup Score")
ax.set_xlabel("Pitcher")
ax.legend(title="Batter Stands")
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.savefig('../reports/figures/matchup_by_handedness.png',
            dpi=150, bbox_inches='tight')
plt.show()

## Connecting Deviation Cost to Exploitability

In [ ]:
# Does higher deviation cost correlate with higher matchup scores?
# This validates that your matchup engine is capturing real vulnerability

exploitability = (matchup_df.groupby('pitcher_name')['matchup_score']
                  .mean()
                  .reset_index()
                  .rename(columns={'matchup_score': 'avg_exploitability'}))

deviation_vs_exploit = pitcher_profiles.merge(
    exploitability, on='pitcher_name'
)

fig, ax = plt.subplots(figsize=(10, 7))

ax.scatter(deviation_vs_exploit['deviation_advantage'],
           deviation_vs_exploit['avg_exploitability'],
           s=100, color='steelblue', zorder=5)

for _, row in deviation_vs_exploit.iterrows():
    ax.annotate(row['pitcher_name'],
               (row['deviation_advantage'], row['avg_exploitability']),
               textcoords="offset points",
               xytext=(8, 4), fontsize=8)

# Trend line
z = np.polyfit(deviation_vs_exploit['deviation_advantage'],
               deviation_vs_exploit['avg_exploitability'], 1)
p = np.poly1d(z)
x_line = np.linspace(deviation_vs_exploit['deviation_advantage'].min(),
                      deviation_vs_exploit['deviation_advantage'].max(), 100)
ax.plot(x_line, p(x_line), 'r--', alpha=0.7, linewidth=2,
        label='Trend')

ax.axvline(x=0, color='black', linewidth=1, linestyle='--', alpha=0.5)
ax.set_xlabel("Deviation Advantage % (negative = model is better)",
              fontsize=11)
ax.set_ylabel("Average Batter Exploitability Score", fontsize=11)
ax.set_title("OffScript — Does Deviation Cost Predict Exploitability?\n"
             "Pitchers who deviate more costly should be more exploitable",
             fontsize=12)
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/deviation_vs_exploitability.png',
            dpi=150, bbox_inches='tight')
plt.show()

# Calculate correlation
corr = deviation_vs_exploit['deviation_advantage'].corr(
    deviation_vs_exploit['avg_exploitability']
)
print(f"Correlation between deviation cost and exploitability: {corr:.3f}")
print("\nIf negative — pitchers with worse deviation outcomes are")
print("more exploitable, validating the matchup engine")

## Save Final Matchup Data

In [ ]:
matchup_df.to_parquet(
    '../data/processed/matchup_scores.parquet',
    index=False
)

# Save top 5 matchups per pitcher as a summary
top_matchups = (matchup_df.sort_values('matchup_score', ascending=False)
                .groupby('pitcher_name')
                .head(5)
                .reset_index(drop=True))

top_matchups.to_parquet(
    '../data/processed/top_matchups_per_pitcher.parquet',
    index=False
)

print("=== Phase 4 Data Saved ===")
print(f"matchup_scores.parquet: {len(matchup_df)} records")
print(f"top_matchups_per_pitcher.parquet: {len(top_matchups)} records")

## Phase 4 Summary

## Phase 4 Complete — Matchup Analysis Summary

### Dataset
- **Pitchers analyzed:** 14
- **Unique batters profiled:** 415
- **Matchup combinations scored:** 5,786

### Key Findings

**Most Universally Exploitable Pitchers**
Chris Sale (39.37) and Dylan Cease (39.30) are the most exploitable
pitchers in the dataset, directly consistent with their Phase 3
deviation penalty scores of -10.74% and -2.06% respectively.

**Least Exploitable Pitchers**
Corbin Burnes (17.16), Framber Valdez (18.97), and Logan Webb (20.82)
are the hardest pitchers to exploit through deviation patterns —
consistent with Burnes being the only pitcher where deviating
outperformed the model recommendation in Phase 3.

**Handedness Patterns**
Right-handed batters exploit deviation patterns more effectively
against 10 of 14 pitchers. Spencer Strider shows the largest
handedness gap (-5.75) favoring right-handed batters. Nestor Cortes
(+2.85) and Max Scherzer (+2.45) are the only pitchers where
left-handed batters show meaningfully higher exploitation potential.

**Most Universally Dangerous Batter**
Justin Turner appears as a top-5 exploitation candidate against
5 of 14 pitchers, reflecting his contact-oriented approach against
pitch types that deviation-prone pitchers tend to throw in
suboptimal situations.

### Engine Validation
The matchup engine produces a correlation of -0.363 between pitcher
deviation cost and average batter exploitability. The negative
direction confirms the engine is correctly identifying that pitchers
with more costly deviation patterns are more exploitable. The moderate
magnitude reflects that pitch selection is one meaningful factor among
several that determine matchup outcomes.

### How the Matchup Score Works
Each score combines batter whiff rate and hit rate against the pitch
types a pitcher throws when deviating from model recommendations,
weighted 60/40 respectively and scaled by deviation frequency.
Higher scores indicate batters better positioned to exploit a
pitcher's deviation tendencies.